# Inference speed-ups for one request

How much faster can an LLM answer one request on one GPU? This notebook measures four
standard techniques on Qwen3-4B. Where a technique fits in a few lines, it is first built
from scratch so you can see how it works, then used through its switch in Hugging Face
`transformers`.

| Section | Technique | Idea | Same output? |
|---|---|---|---|
| A1 | KV cache | Store the keys and values of past tokens instead of recomputing them | Yes |
| A2 | Static cache + `torch.compile` | Fixed tensor shapes, so many small GPU calls can be merged | Yes |
| A3 | Flash / memory-efficient attention | Compute attention in tiles; the N x N score matrix is never stored | Yes |
| A4 | Speculative decoding | A small model guesses tokens; the big model checks several in one pass | Yes |

"Same output" means the same tokens, up to 16-bit rounding (A1 shows why rounding can
matter).

## Two phases of generation

1. **Prefill**: the model reads the whole prompt in one forward pass. Many tokens at once,
   so the GPU is busy computing.
2. **Decode**: the model produces new tokens one at a time, one forward pass per token.

Decode takes most of the time for a normal chat answer. It is slow for a simple reason:
to produce one token, the model reads **every weight** from GPU memory once. At batch
size 1 the GPU spends most of its time waiting for that read, not computing. That gives a
speed limit:

    tokens per second <= memory bandwidth / size of the weights in bytes

Example: Qwen3-4B in 16-bit is about 8 GB. A T4 reads about 320 GB/s, so at most ~40
tokens/s. An H200 reads about 4800 GB/s: at most ~600 tokens/s.

Each technique attacks part of the cost:
- A1 avoids repeated work.
- A2 removes overhead between GPU calls.
- A3 moves fewer bytes inside attention (matters most for long prompts).
- A4 gets several tokens out of one read of the weights.

## How to read the results

- Speed is **tokens per second** (tok/s): new tokens / total generation time.
- Every technique is compared with a **baseline measured right before it**, in the same
  section, on the same prompt.
- Each speed is the **median** of several runs, with a 95% confidence interval. With 3-5
  runs the interval is rough: it only shows run-to-run noise on this machine.
- The summary at the end lists every comparison as "x better" than its baseline.

## Assumptions and limits

- One NVIDIA GPU with 16 GB or more: a Colab T4 or a rented GPU machine. Newer GPUs
  (A100, L4, H100, H200) add bf16 and FlashAttention-2; on a T4 the code uses fp16 and
  skips what the GPU cannot run.
- Greedy decoding (always take the most likely token), batch size 1, every run produces
  the same number of tokens.
- Serving many users at once (continuous batching, paged attention) is not covered.
- Absolute numbers depend on the GPU and library versions. Compare within one run, not
  across machines.
- `generate()` has Python overhead per token, so dedicated engines (vLLM, TensorRT-LLM,
  llama.cpp) gain more from the same ideas.
- Run time: roughly 20-40 minutes on a T4, plus about 9 GB of model downloads (estimate,
  not measured).

## 0. Setup

You need a Python environment with PyTorch built for CUDA. The install cell adds the
other packages.

- **Colab**: Runtime > Change runtime type > GPU. Run the install cell, restart the
  session (Runtime > Restart session), then continue from the imports cell. The restart
  is needed because Colab has an older `transformers` already loaded.
- **Your own GPU machine**: run the install cell once, restart the kernel, then continue
  from the imports cell.

When done, shut the GPU down: Colab: Runtime > Disconnect and delete runtime. Rented
machine: stop it in the provider's console, it is billed per hour.

In [ ]:
# PyTorch with CUDA must already be installed. Versions this notebook was written for
%pip install -q "transformers==5.19.0" "accelerate==1.15.0" pandas matplotlib

In [ ]:
import gc
import logging
import math
import os
import sys
import time
from collections.abc import Callable
from dataclasses import dataclass
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DynamicCache,
    GenerationConfig,
    PreTrainedModel,
    set_seed,
)

# Our messages go to the notebook output as plain text
log = logging.getLogger("inference")
log.setLevel(logging.INFO)
log.handlers[:] = [logging.StreamHandler(sys.stdout)]
log.propagate = False

In [ ]:
assert torch.cuda.is_available(), "No GPU found. Colab: Runtime > Change runtime type"

DEVICE = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(DEVICE)
# bf16 needs compute capability 8.0+ (Ampere and newer); a T4 (7.5) uses fp16
IS_AMPERE_PLUS = torch.cuda.get_device_capability(DEVICE) >= (8, 0)
DTYPE = torch.bfloat16 if IS_AMPERE_PLUS else torch.float16
GB = 1e9

vram_gb = torch.cuda.get_device_properties(DEVICE).total_memory / GB
log.info(f"GPU:   {GPU_NAME}, {vram_gb:.0f} GB")
log.info(f"dtype: {DTYPE}")
log.info(f"torch {torch.__version__}, CUDA {torch.version.cuda}")

## 1. Settings and prompts

All settings are in one place. To try other models, set environment variables before
this cell, e.g. `%env INFERENCE_TARGET_MODEL=Qwen/Qwen3-1.7B`. The draft model (A4) must
use the same tokenizer as the target. Lower `n_repeat` for a faster but noisier run.

Two prompts:
- **open**: explain something in free text. The answer has little in common with the
  prompt: the normal chat case.
- **grounded**: add type hints to given code. The answer copies most of the prompt, which
  helps speculative decoding (A4).

In [ ]:
@dataclass(frozen=True)
class Config:
    """All settings of the notebook."""

    # The draft model proposes tokens for the target in A4: same tokenizer needed
    target_model: str = os.environ.get("INFERENCE_TARGET_MODEL", "Qwen/Qwen3-4B")
    draft_model: str = os.environ.get("INFERENCE_DRAFT_MODEL", "Qwen/Qwen3-0.6B")
    seed: int = 42
    # Every generation produces exactly this many new tokens
    n_new_tokens: int = 256
    # Timed runs per configuration; fewer for slow ones (no cache, long prompts)
    n_repeat: int = 5
    n_repeat_slow: int = 3
    # A1: output lengths for the loops with and without cache
    cache_sweep: tuple[int, ...] = (32, 64, 128, 256)
    # A3: sequence lengths for attention alone and for the full model (prefill)
    attn_seq_lens: tuple[int, ...] = (1024, 2048, 4096, 8192, 16384)
    prefill_lens: tuple[int, ...] = (1024, 2048, 4096, 8192)
    # A4: draft tokens per round; tokens copied from the prompt per round
    spec_k: int = 5
    prompt_lookup_tokens: int = 10


cfg = Config()
set_seed(cfg.seed)
RNG = np.random.default_rng(cfg.seed)  # for the confidence intervals
display(pd.Series(vars(cfg), name="value").to_frame())

In [ ]:
OPEN_PROMPT = (
    "Explain how a refrigerator keeps food cold. Cover the refrigerant cycle, the "
    "compressor, the condenser and the evaporator, step by step."
)

CODE_SNIPPET = '''def moving_average(values, window):
    if window <= 0:
        raise ValueError("window must be positive")
    result = []
    total = 0.0
    for i, value in enumerate(values):
        total += value
        if i >= window:
            total -= values[i - window]
        if i >= window - 1:
            result.append(total / window)
    return result


def normalize(values):
    low = min(values)
    high = max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value - low) / (high - low) for value in values]


def zscore(values):
    n = len(values)
    mean = sum(values) / n
    variance = sum((value - mean) ** 2 for value in values) / n
    std = variance ** 0.5
    return [(value - mean) / std for value in values]
'''

GROUNDED_PROMPT = (
    "Add Python type hints to every function below. Return the full code and nothing "
    "else.\n\n" + CODE_SNIPPET
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.target_model)


def chat_input_ids(prompt: str) -> torch.Tensor:
    """Wrap a prompt in the model's chat template and turn it into token ids."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3: answer directly; other templates ignore it
    )
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids
    return ids.to(DEVICE)


PROMPTS = {
    "open": chat_input_ids(OPEN_PROMPT),
    "grounded": chat_input_ids(GROUNDED_PROMPT),
}
open_ids = PROMPTS["open"]
for name, ids in PROMPTS.items():
    log.info(f"{name} prompt: {ids.shape[1]} tokens")

## 2. How we measure

Timing on a GPU has three traps:
- **The GPU runs asynchronously.** A CUDA call returns before the GPU has finished. We
  call `torch.cuda.synchronize()` before reading the clock; otherwise we would time the
  launch, not the work.
- **The first calls are slow.** They pay for CUDA setup, kernel selection and
  compilation. We run a warm-up call first and throw its time away.
- **Every run must do the same work.** Greedy decoding with `min_new_tokens =
  max_new_tokens` blocks the end-of-text token, so every run generates exactly
  `n_new_tokens`.

Peak memory is `torch.cuda.max_memory_allocated()`: memory held by PyTorch tensors.
`nvidia-smi` shows more (allocator cache, CUDA context).

GPU clocks change as the card heats up, so every section measures its own baseline right
before the technique.

In [ ]:
def free_memory() -> None:
    """Free unused GPU memory and restart the peak-memory counter."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def time_calls(
    fn: Callable[[], object], n_repeat: int, n_warmup: int = 1
) -> np.ndarray:
    """Seconds per call of fn for n_repeat calls, after n_warmup discarded calls."""
    for _ in range(n_warmup):
        fn()
    seconds = np.empty(n_repeat)
    for i in range(n_repeat):
        torch.cuda.synchronize()  # nothing from earlier calls is still running
        start = time.perf_counter()
        fn()
        torch.cuda.synchronize()  # wait until the GPU has really finished
        seconds[i] = time.perf_counter() - start
    return seconds


# Errors SDPA raises when a backend cannot run on this GPU, dtype or shape
NO_KERNEL_MESSAGES = ("no available kernel", "no viable backend")


def try_time_calls(
    fn: Callable[[], object], n_repeat: int, n_warmup: int = 1
) -> tuple[str, np.ndarray | None]:
    """time_calls(), but returns a status on out-of-memory or a missing kernel."""
    try:
        return "ok", time_calls(fn, n_repeat, n_warmup)
    except torch.OutOfMemoryError:
        status = "OOM"
    except RuntimeError as err:
        if not any(text in str(err).lower() for text in NO_KERNEL_MESSAGES):
            raise
        status = "not supported"
    free_memory()  # after the except block, so the failed call's tensors can be freed
    return status, None


def median_ci(x: np.ndarray, n_boot: int = 2000) -> tuple[float, float]:
    """95% bootstrap confidence interval of the median of x."""
    resamples = x[RNG.integers(0, len(x), size=(n_boot, len(x)))]
    low, high = np.quantile(np.median(resamples, axis=1), [0.025, 0.975])
    return float(low), float(high)


def token_agreement(a: torch.Tensor, b: torch.Tensor) -> tuple[float, int | None]:
    """Share of positions with the same token, and the first position that differs."""
    a, b = a.flatten().cpu(), b.flatten().cpu()
    n = min(len(a), len(b))
    same = a[:n] == b[:n]
    first_diff = None if bool(same.all()) else int((~same).nonzero()[0])
    return float(same.float().mean()), first_diff

In [ ]:
def generate(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    n_new: int = cfg.n_new_tokens,
    **options: object,
) -> torch.Tensor:
    """Greedy generate() that always makes exactly n_new tokens; returns only those."""
    out = model.generate(
        input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=n_new,
        min_new_tokens=n_new,  # blocks end-of-text: every run does the same work
        do_sample=False,  # greedy: always the most likely token
        **options,
    )
    return out[0, input_ids.shape[1]:]


@dataclass(frozen=True)
class Run:
    """Speeds of repeated generate() calls with one configuration."""

    tok_s: np.ndarray  # tokens per second, one value per timed run
    peak_gb: float  # peak GPU memory held by tensors


def measure_generate(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    n_repeat: int = cfg.n_repeat,
    n_warmup: int = 1,
    **options: object,
) -> Run:
    """Time generate() with the given options: tok/s of every run and peak memory."""
    free_memory()
    call = partial(generate, model, input_ids, **options)
    seconds = time_calls(call, n_repeat, n_warmup)
    return Run(cfg.n_new_tokens / seconds, torch.cuda.max_memory_allocated() / GB)


def speed_table(runs: dict[str, Run]) -> pd.DataFrame:
    """Median tok/s with 95% CI, speed-up and peak memory; first row = baseline."""
    baseline = np.median(next(iter(runs.values())).tok_s)
    rows = []
    for name, run in runs.items():
        low, high = median_ci(run.tok_s)
        rows.append({
            "config": name,
            "tok/s": np.median(run.tok_s),
            "tok/s 95% CI": f"{low:.1f} - {high:.1f}",
            "speed-up": np.median(run.tok_s) / baseline,
            "peak GB": run.peak_gb,
        })
    return pd.DataFrame(rows).set_index("config").round(2)

In [ ]:
SUMMARY: list[dict] = []  # one row per comparison, shown at the end


def add_to_summary(
    section: str,
    technique: str,
    metric: str,
    baseline: float,
    new: float,
    higher_is_better: bool,
) -> None:
    """Store one comparison as 'x better than the baseline' (2 = twice as good)."""
    x_better = new / baseline if higher_is_better else baseline / new
    SUMMARY.append({
        "section": section,
        "technique": technique,
        "metric": metric,
        "baseline": baseline,
        "new": new,
        "x better": x_better,
    })
    log.info(f"{section} {technique}: {metric} {baseline:.3g} -> {new:.3g}, "
             f"{x_better:.2f}x better")


def add_runs_to_summary(section: str, runs: dict[str, Run], label: str = "") -> None:
    """Add the median tok/s of every configuration against the first one (baseline)."""
    names = list(runs)
    baseline = np.median(runs[names[0]].tok_s)
    for name in names[1:]:
        new = np.median(runs[name].tok_s)
        add_to_summary(section, name + label, "tok/s", baseline, new, True)


# Chart style: light grid, no top/right frame. Line colors come from a
# colorblind-checked palette, used in a fixed order. Summary: blue = better, red = worse
COLORS = ("#2a78d6", "#eb6834", "#1baf7a")
BETTER, WORSE = "#2a78d6", "#e34948"
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e1e0d9",
    "axes.titlecolor": "#0b0b0b",
    "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "legend.frameon": False,
})


def plot_lines(ax: plt.Axes, df: pd.DataFrame, x: str, y: str, group: str) -> None:
    """One line per group, colors in a fixed order; groups without data are skipped."""
    for color, (name, part) in zip(COLORS, df.groupby(group, sort=False)):
        part = part.dropna(subset=[y])
        if not part.empty:
            ax.plot(part[x], part[y], color=color, marker="o", label=name)
    ax.legend()

# Part A: lower latency for one request

## A0. Load the model and compute the speed limit

The weights are loaded in 16-bit. Attention uses SDPA, the `transformers` default (A3
explains it). Qwen3 ships with sampling settings (temperature 0.6, top-p 0.95); we
replace them with an empty generation config, which means greedy decoding.

Speed limit: every new token reads all weights once, so decoding cannot be faster than
memory bandwidth / weight bytes. Real decoding stays below it (cache reads, kernel
launches, Python), but the limit shows how much room there is.

In [ ]:
# Spec-sheet memory bandwidth in GB/s (approximate). The first matching name wins, so
# "L40S" comes before "L4" and "H100 PCIe" before "H100". Unknown GPU: add it here
GPU_BANDWIDTH_GBS = {
    "H200": 4800, "B200": 8000, "H100 PCIe": 2000, "H100": 3350, "A100": 1555,
    "L40S": 864, "A10G": 600, "L4": 300, "T4": 320,
}


def load_model(name: str) -> PreTrainedModel:
    """Load a model on the GPU in 16-bit and switch it to greedy generation."""
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=DTYPE, attn_implementation="sdpa", device_map={"": DEVICE}
    )
    model.eval()
    shipped = model.generation_config
    pad_id = shipped.pad_token_id
    # A generation config without sampling settings means greedy decoding
    model.generation_config = GenerationConfig(
        bos_token_id=shipped.bos_token_id,
        eos_token_id=shipped.eos_token_id,
        pad_token_id=pad_id if pad_id is not None else tokenizer.pad_token_id,
    )
    return model

In [ ]:
target = load_model(cfg.target_model)
config = target.config
HEAD_DIM = getattr(config, "head_dim", None) or (
    config.hidden_size // config.num_attention_heads
)
weight_bytes = target.get_memory_footprint()
bandwidth_gbs = next(
    (gbs for name, gbs in GPU_BANDWIDTH_GBS.items() if name in GPU_NAME), math.nan
)
SPEED_LIMIT = bandwidth_gbs * 1e9 / weight_bytes  # tok/s

n_params = sum(p.numel() for p in target.parameters())
log.info(f"Model:       {cfg.target_model}, {n_params / 1e9:.2f} B parameters")
log.info(f"Layers:      {config.num_hidden_layers}")
log.info(f"Heads:       {config.num_attention_heads} query, "
         f"{config.num_key_value_heads} key/value, head dim {HEAD_DIM}")
log.info(f"Weights:     {weight_bytes / GB:.2f} GB")
log.info(f"Speed limit: {SPEED_LIMIT:.0f} tok/s "
         f"({bandwidth_gbs:.0f} GB/s / {weight_bytes / GB:.1f} GB)")

## A1. KV cache

The model writes one token at a time. For each new token, attention compares it with all
earlier tokens, using their **keys** (K) and **values** (V). The keys and values of
earlier tokens never change, so they can be stored and reused:

- **Without cache**: every step runs the model over the whole sequence again. Step t
  processes t tokens, so n new tokens cost O(n^2) work.
- **With KV cache**: every step runs the model over the newest token only, and adds its
  keys and values to the cache. O(n) work, paid for with memory.

From scratch first: two greedy loops that differ only in the cache.

`logits_to_keep=1`: the output layer turns every position into 151k vocabulary scores.
We only need the scores of the last position. Without this option the no-cache loop
would also pay for scores at all earlier positions, and the cache would look better than
it is.

In [ ]:
@torch.inference_mode()
def greedy_no_cache(
    model: PreTrainedModel, input_ids: torch.Tensor, n_new: int
) -> torch.Tensor:
    """Greedy decoding that runs the model over the whole sequence at every step."""
    ids = input_ids
    for _ in range(n_new):
        logits = model(ids, use_cache=False, logits_to_keep=1).logits[:, -1]
        next_id = logits.argmax(dim=-1, keepdim=True)  # greedy: most likely token
        ids = torch.cat([ids, next_id], dim=1)  # the input grows by one token
    return ids[:, input_ids.shape[1]:]


@torch.inference_mode()
def greedy_with_cache(
    model: PreTrainedModel, input_ids: torch.Tensor, n_new: int
) -> torch.Tensor:
    """Greedy decoding that feeds only the newest token and reuses cached K/V."""
    cache = DynamicCache(config=model.config)  # grows by one position per step
    step_ids = input_ids  # first step: the whole prompt (prefill)
    new_ids = []
    for _ in range(n_new):
        out = model(step_ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
        step_ids = out.logits[:, -1].argmax(dim=-1, keepdim=True)  # next input: 1 token
        new_ids.append(step_ids)
    return torch.cat(new_ids, dim=1)

Both loops should produce the same tokens. In 16-bit, a matrix product over 300 tokens
and one over 1 token round slightly differently. After many tokens, greedy decoding can
then pick the other of two nearly equal candidates. A difference late in the sequence is
rounding; a difference in the first few tokens would be a bug.

This cell also warms up both loops.

In [ ]:
no_cache_ids = greedy_no_cache(target, open_ids, 64)
cache_ids = greedy_with_cache(target, open_ids, 64)
share, first_diff = token_agreement(no_cache_ids, cache_ids)
log.info(f"Same tokens with and without cache: {share:.0%} "
         f"(first difference at position {first_diff})")
log.info(tokenizer.decode(cache_ids[0]))

Time against the number of new tokens: it should grow quadratically without the cache and
linearly with it.

In [ ]:
LOOPS = {"no cache": greedy_no_cache, "KV cache": greedy_with_cache}

rows = []
for n in cfg.cache_sweep:
    for name, loop in LOOPS.items():
        call = partial(loop, target, open_ids, n)
        seconds = time_calls(call, cfg.n_repeat_slow, n_warmup=0)  # warmed up above
        rows.append({"loop": name, "new tokens": n, "seconds": np.median(seconds)})
sweep = pd.DataFrame(rows)
display(sweep.pivot(index="new tokens", columns="loop", values="seconds").round(2))

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
plot_lines(ax, sweep, "new tokens", "seconds", "loop")
ax.set(xlabel="new tokens", ylabel="seconds (median)",
       title="Generation time, from-scratch loops")
plt.show()

n_max = cfg.cache_sweep[-1]
at_max = sweep[sweep["new tokens"] == n_max].set_index("loop")["seconds"]
add_to_summary("A1", "KV cache, from scratch", f"seconds for {n_max} tokens",
               at_max["no cache"], at_max["KV cache"], higher_is_better=False)

The same switch in `generate()`: `use_cache=False` against the default (KV cache on).

In [ ]:
a1_runs = {
    "no cache": measure_generate(
        target, open_ids, n_repeat=cfg.n_repeat_slow, use_cache=False
    ),
    "KV cache (default)": measure_generate(target, open_ids),
}
display(speed_table(a1_runs))
add_runs_to_summary("A1", a1_runs, ", generate()")

cached = np.median(a1_runs["KV cache (default)"].tok_s)
log.info(f"With KV cache: {cached:.0f} tok/s = {cached / SPEED_LIMIT:.0%} "
         f"of the speed limit ({SPEED_LIMIT:.0f} tok/s)")

### What the cache costs

For every token, every layer stores one key vector and one value vector per KV head:

    bytes per token = 2 x layers x KV heads x head dim x bytes per value

Qwen3 uses **grouped-query attention (GQA)**: several query heads share one KV head.
Qwen3-4B has 32 query heads and 8 KV heads, so its cache is 4x smaller than if every
query head had its own (multi-head attention).

For long contexts or many users at once, the cache, not the weights, fills the GPU.

In [ ]:
def kv_bytes_per_token(kv_heads: int) -> int:
    """KV cache bytes per token: one key and one value per layer and KV head."""
    return 2 * config.num_hidden_layers * kv_heads * HEAD_DIM * DTYPE.itemsize


per_token = kv_bytes_per_token(config.num_key_value_heads)  # GQA, as in Qwen3
per_token_mha = kv_bytes_per_token(config.num_attention_heads)  # one KV head per query
log.info(f"KV cache per token: {per_token / 1e3:.0f} KB "
         f"(without GQA: {per_token_mha / 1e3:.0f} KB)")

kv_table = pd.DataFrame([
    {
        "context tokens": context,
        "users (batch)": batch,
        "KV cache GB": per_token * context * batch / GB,
        "KV cache GB without GQA": per_token_mha * context * batch / GB,
        "weights GB": weight_bytes / GB,
    }
    for context in (4096, 32768)
    for batch in (1, 8)
])
display(kv_table.round(1))

## A2. Static cache and `torch.compile`

At batch size 1, every decode step launches hundreds of small GPU functions ("kernels"),
several per layer. Each one finishes quickly, so the time Python and the CUDA driver
spend *starting* them is a large part of the step.

`torch.compile` can merge kernels and record the whole step once as a **CUDA graph**,
then replay it with a single launch. That only works if tensor shapes never change. A
normal (dynamic) cache grows by one position per step, so its shape changes every step.
A **static cache** reserves space for the maximum length up front, so shapes stay fixed.

`transformers` compiles the decode step automatically with
`cache_implementation="static"`. `disable_compile=True` turns compilation off, so we can
separate the two effects:
1. dynamic cache (baseline)
2. static cache, no compile
3. static cache + compile

Compilation happens once per shape (prompt length + output length) and can take a minute
or more. We time it separately. A new prompt length means a new compilation, so this
pays off for servers with fixed shapes, not for one-off scripts.

In [ ]:
a2_runs = {
    "dynamic cache (baseline)": measure_generate(target, open_ids),
    "static cache, no compile": measure_generate(
        target, open_ids, cache_implementation="static", disable_compile=True
    ),
}

# The first call with compilation on traces and compiles the decode step: time it alone
try:
    first_call = partial(generate, target, open_ids, cache_implementation="static")
    compile_s = time_calls(first_call, n_repeat=1, n_warmup=0)[0]
    a2_runs["static cache + torch.compile"] = measure_generate(
        target, open_ids, n_warmup=2, cache_implementation="static"
    )
except Exception as err:  # compile support depends on GPU, driver and Triton version
    compile_s = math.nan
    log.warning(f"torch.compile failed, skipped: {type(err).__name__}: {err}")

display(speed_table(a2_runs))
add_runs_to_summary("A2", a2_runs)

In [ ]:
if "static cache + torch.compile" in a2_runs:
    base = np.median(a2_runs["dynamic cache (baseline)"].tok_s)
    compiled = np.median(a2_runs["static cache + torch.compile"].tok_s)
    # First call = compilation + one normal generation
    one_off_s = compile_s - cfg.n_new_tokens / compiled
    log.info(f"Compilation: ~{one_off_s:.0f} s, once per prompt length + output length")
    if compiled > base:
        saved_per_token = 1 / base - 1 / compiled
        log.info(f"Pays off after ~{one_off_s / saved_per_token:,.0f} generated tokens "
                 "with the same shapes")

# Drop the compiled graphs and their memory before the next section
torch.compiler.reset()
free_memory()

## A3. Flash and memory-efficient attention

Attention compares every token with every earlier token. The textbook ("eager") way:
1. Compute the score matrix S = Q K^T: N x N numbers per head, for N tokens.
2. Write S to GPU memory, read it back for the softmax, write it again.
3. Multiply by V.

For long sequences S is huge. Qwen3-4B has 32 heads, so at 8192 tokens S takes
32 x 8192^2 x 2 bytes = 4.3 GB, for one layer. The GPU spends its time moving S around.

**FlashAttention** computes the same result in small tiles that fit in the GPU's fast
on-chip memory (SRAM), with a running softmax. S is never written to GPU memory, so
memory grows with N instead of N^2 and far fewer bytes move.

PyTorch has this built in as `scaled_dot_product_attention` (SDPA), with several
backends:

| Backend | How | GPUs |
|---|---|---|
| memory-efficient | Tiled, xFormers-style | All CUDA GPUs, incl. T4 |
| flash | FlashAttention-2 | Ampere and newer (not T4) |

`transformers` uses SDPA by default; `attn_implementation="eager"` is the textbook
version.

Three measurements:
1. Attention alone, for one layer, at growing sequence lengths.
2. The full model reading a long prompt (prefill).
3. Normal generation with a short prompt.

First a check: every kernel must give the same result as the textbook formula, up to
16-bit rounding.

In [ ]:
def eager_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Causal attention the textbook way: builds the full N x N score matrix."""
    n = q.shape[-2]
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])  # (1, heads, N, N)
    future = torch.ones(n, n, dtype=torch.bool, device=q.device).triu(1)
    scores = scores.masked_fill(future, float("-inf"))  # no looking at later tokens
    weights = torch.softmax(scores.float(), dim=-1)  # in fp32, as transformers does
    return weights.to(q.dtype) @ v


def sdpa(
    backend: SDPBackend,
) -> Callable[[torch.Tensor, torch.Tensor, torch.Tensor], torch.Tensor]:
    """PyTorch's built-in attention, restricted to one backend."""

    def attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        """SDPA call that may only use the chosen backend."""
        with sdpa_kernel(backend):
            return F.scaled_dot_product_attention(q, k, v, is_causal=True)

    return attention


EAGER = "eager (from scratch)"
KERNELS = {
    EAGER: eager_attention,
    "SDPA memory-efficient": sdpa(SDPBackend.EFFICIENT_ATTENTION),
    "SDPA flash": sdpa(SDPBackend.FLASH_ATTENTION),
}


def random_qkv(n: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Random queries, keys and values for one layer: shape (1, heads, n, head dim)."""
    gen = torch.Generator(device=DEVICE).manual_seed(cfg.seed)
    shape = (1, config.num_attention_heads, n, HEAD_DIM)
    q, k, v = (
        torch.randn(shape, device=DEVICE, dtype=DTYPE, generator=gen) for _ in range(3)
    )
    return q, k, v


q, k, v = random_qkv(cfg.attn_seq_lens[0])
reference = eager_attention(q, k, v)
for name, kernel in KERNELS.items():
    if name == EAGER:
        continue
    try:
        diff = (kernel(q, k, v) - reference).abs().max().item()
        log.info(f"{name:<22} max difference from eager: {diff:.1e}")
    except RuntimeError as err:
        log.info(f"{name:<22} not available here: {str(err).splitlines()[0]}")
del q, k, v, reference
free_memory()

Attention alone, one layer, at growing sequence lengths. "extra GB" is the memory the
attention call needs on top of its inputs. A kernel that runs out of memory (OOM) is not
tried at longer lengths. The model stays loaded, so the free memory is what a real model
would have left.

In [ ]:
attn_rows = []
failed = {}  # kernel -> status, once it failed at a shorter length
for n in cfg.attn_seq_lens:
    q, k, v = random_qkv(n)
    for name, kernel in KERNELS.items():
        row = {"kernel": name, "seq len": n, "status": failed.get(name, "ok")}
        if name not in failed:
            free_memory()
            before = torch.cuda.memory_allocated()
            status, seconds = try_time_calls(partial(kernel, q, k, v), cfg.n_repeat)
            row["status"] = status
            if seconds is None:
                failed[name] = status
            else:
                row["ms"] = np.median(seconds) * 1e3
                row["extra GB"] = (torch.cuda.max_memory_allocated() - before) / GB
        attn_rows.append(row)
    del q, k, v
free_memory()

attn = pd.DataFrame(attn_rows)
for value in ("ms", "extra GB", "status"):
    table = attn.pivot(index="seq len", columns="kernel", values=value)
    display(table[list(KERNELS)].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
panels = (("ms", "Attention time, one layer (ms)"),
          ("extra GB", "Attention memory, one layer (GB); lines end at OOM"))
for ax, (value, title) in zip(axes, panels):
    plot_lines(ax, attn, "seq len", value, "kernel")
    ax.set(xscale="log", yscale="log", xlabel="sequence length (tokens)", title=title)
    ax.set_xticks(cfg.attn_seq_lens, [str(n) for n in cfg.attn_seq_lens])
    ax.xaxis.set_minor_locator(plt.NullLocator())
plt.show()

# Summary: eager against the fastest tiled kernel, at the longest length eager managed
done = attn.dropna(subset=["ms"])
n_cmp = int(done.loc[done["kernel"] == EAGER, "seq len"].max())
at_n = done[done["seq len"] == n_cmp].set_index("kernel")
best = at_n.drop(index=EAGER)["ms"].idxmin()
technique = f"{best}, attention alone, {n_cmp} tokens"
add_to_summary("A3", technique, "time (ms)",
               at_n.loc[EAGER, "ms"], at_n.loc[best, "ms"], higher_is_better=False)
add_to_summary("A3", technique, "extra memory (GB)",
               at_n.loc[EAGER, "extra GB"], at_n.loc[best, "extra GB"],
               higher_is_better=False)

The full model reading a prompt of N tokens once (prefill), eager against SDPA. The
prompt is random token ids: the content does not change the cost. "extra GB" is the peak
memory above the loaded weights: activations, attention scores and the KV cache of the
prompt.

In [ ]:
@torch.inference_mode()
def prefill(ids: torch.Tensor) -> None:
    """Read a prompt once into a fresh KV cache, as generate() does before token 1."""
    target(ids, use_cache=True, logits_to_keep=1)


prefill_rows = []
for impl in ("eager", "sdpa"):
    target.set_attn_implementation(impl)
    for n in cfg.prefill_lens:
        gen = torch.Generator().manual_seed(cfg.seed)
        ids = torch.randint(0, config.vocab_size, (1, n), generator=gen).to(DEVICE)
        free_memory()
        before = torch.cuda.memory_allocated()
        status, seconds = try_time_calls(partial(prefill, ids), cfg.n_repeat_slow)
        row = {"attention": impl, "prompt tokens": n, "status": status}
        if seconds is None:
            prefill_rows.append(row)
            break  # longer prompts would fail too
        row["prefill tok/s"] = n / np.median(seconds)
        row["extra GB"] = (torch.cuda.max_memory_allocated() - before) / GB
        prefill_rows.append(row)
target.set_attn_implementation("sdpa")
free_memory()

prefill_df = pd.DataFrame(prefill_rows)
display(prefill_df.pivot(index="prompt tokens", columns="attention",
                         values=["prefill tok/s", "extra GB", "status"]).round(2))

# Summary: the longest prompt both implementations managed
both = prefill_df.pivot(index="prompt tokens", columns="attention",
                        values=["prefill tok/s", "extra GB"]).dropna()
if not both.empty:
    n_cmp = int(both.index.max())
    row = both.loc[n_cmp]
    technique = f"SDPA, prefill {n_cmp} tokens"
    add_to_summary("A3", technique, "prefill tok/s", row["prefill tok/s", "eager"],
                   row["prefill tok/s", "sdpa"], higher_is_better=True)
    add_to_summary("A3", technique, "extra memory (GB)", row["extra GB", "eager"],
                   row["extra GB", "sdpa"], higher_is_better=False)

Normal generation with the short open prompt. Each decode step attends over only a few
hundred positions, so attention is a small part of the step: expect little difference.

In [ ]:
a3_runs = {}
for impl in ("eager", "sdpa"):
    target.set_attn_implementation(impl)
    a3_runs[f"{impl} attention"] = measure_generate(target, open_ids)
target.set_attn_implementation("sdpa")

display(speed_table(a3_runs))
add_runs_to_summary("A3", a3_runs, ", short-prompt generation")

## A4. Speculative decoding

At batch size 1, a forward pass over 1 token and one over 6 tokens take about the same
time: both read all weights once, and the reading dominates. Speculative decoding uses
this:

1. A small **draft** model proposes k tokens, one at a time. This is cheap.
2. The big **target** model checks all k proposals in one forward pass.
3. Proposals are accepted up to the first one where the target would have chosen a
   different token. Then the target adds its own token: the correction, or a bonus
   token if all k matched.

With greedy decoding the output is exactly what the target would write alone (up to
rounding); only the number of target passes drops.

How many tokens per target pass to expect, if each proposal is accepted with
probability a (Leviathan et al., 2023):

    expected tokens per pass = (1 - a^(k+1)) / (1 - a)
    speed-up ~ expected tokens per pass / (1 + k x c)

c is the cost of one draft token relative to one target token. The draft is not free:
that is why the speed-up is lower than the tokens per pass.

**Prompt lookup decoding** needs no draft model: it copies proposals from the prompt,
from the place where the last few generated tokens appeared before. It helps when the
answer repeats the input (editing code, extracting, quoting), and does nothing for free
writing.

In [ ]:
draft = load_model(cfg.draft_model)
assert draft.config.vocab_size == config.vocab_size, "Draft and target differ in vocab"
log.info(f"Draft model: {cfg.draft_model}, {draft.get_memory_footprint() / GB:.2f} GB")

In [ ]:
@torch.inference_mode()
def speculative_greedy(
    target: PreTrainedModel,
    draft: PreTrainedModel,
    input_ids: torch.Tensor,
    n_new: int,
    k: int,
) -> tuple[torch.Tensor, float, float]:
    """Greedy speculative decoding: new tokens, acceptance rate, tokens per pass."""
    target_cache = DynamicCache(config=target.config)
    draft_cache = DynamicCache(config=draft.config)
    n_prompt = input_ids.shape[1]
    ids = input_ids
    rounds = accepted = rejected = 0
    while ids.shape[1] - n_prompt < n_new:
        # 1. Draft: the small model proposes k tokens, one at a time.
        #    First it reads the tokens its cache has not seen yet
        draft_in = ids[:, draft_cache.get_seq_length():]
        proposals = []
        for _ in range(k):
            out = draft(draft_in, past_key_values=draft_cache, use_cache=True,
                        logits_to_keep=1)
            draft_in = out.logits[:, -1].argmax(dim=-1, keepdim=True)
            proposals.append(draft_in)
        proposals = torch.cat(proposals, dim=1)  # (1, k)

        # 2. Verify: one target pass over its unseen tokens plus all k proposals.
        #    choice[:, i] is the token the target itself would pick at proposal i
        unseen = ids[:, target_cache.get_seq_length():]
        target_in = torch.cat([unseen, proposals], dim=1)
        logits = target(target_in, past_key_values=target_cache, use_cache=True,
                        logits_to_keep=k + 1).logits
        choice = logits.argmax(dim=-1)  # (1, k + 1)

        # 3. Accept proposals up to the first mismatch, then add the target's own token
        n_ok = int((proposals[0] == choice[0, :k]).int().cumprod(dim=0).sum())
        ids = torch.cat([ids, proposals[:, :n_ok], choice[:, n_ok:n_ok + 1]], dim=1)
        rounds += 1
        accepted += n_ok
        rejected += int(n_ok < k)  # a round that stops early has exactly one rejection

        # 4. Forget rejected proposals: both caches keep all tokens but the newest
        for cache in (target_cache, draft_cache):
            excess = cache.get_seq_length() - (ids.shape[1] - 1)
            if excess > 0:
                cache.crop(-excess)  # a negative value removes that many positions

    acceptance_rate = accepted / max(accepted + rejected, 1)
    tokens_per_pass = (accepted + rounds) / rounds
    return ids[:, n_prompt:n_prompt + n_new], acceptance_rate, tokens_per_pass

From scratch: the speculative loop against the plain cached loop from A1, same target,
both prompts. The table compares the measured speed-up with the formula, using the
measured acceptance rate and draft cost.

In [ ]:
def expected_tokens_per_pass(alpha: float, k: int) -> float:
    """Expected tokens per target pass if each proposal is accepted with prob. alpha."""
    return k + 1 if alpha >= 1 else (1 - alpha ** (k + 1)) / (1 - alpha)


n_new, k = cfg.n_new_tokens, cfg.spec_k

# c: cost of one draft token relative to one target token
t_draft = time_calls(partial(greedy_with_cache, draft, open_ids, n_new),
                     cfg.n_repeat_slow)
t_target = time_calls(partial(greedy_with_cache, target, open_ids, n_new),
                      cfg.n_repeat_slow)
draft_cost = np.median(t_draft) / np.median(t_target)
log.info(f"Draft / target time per token: c = {draft_cost:.2f}")

spec_rows = []
for prompt_name, ids in PROMPTS.items():
    reference = greedy_with_cache(target, ids, n_new)
    out, alpha, per_pass = speculative_greedy(target, draft, ids, n_new, k)  # warms up
    plain_call = partial(greedy_with_cache, target, ids, n_new)
    spec_call = partial(speculative_greedy, target, draft, ids, n_new, k)
    t_plain = np.median(time_calls(plain_call, cfg.n_repeat_slow, n_warmup=0))
    t_spec = np.median(time_calls(spec_call, cfg.n_repeat_slow, n_warmup=0))
    expected = expected_tokens_per_pass(alpha, k)
    spec_rows.append({
        "prompt": prompt_name,
        "acceptance rate": alpha,
        "tokens per target pass": per_pass,
        "tokens per pass (formula)": expected,
        "speed-up (formula)": expected / (1 + k * draft_cost),
        "speed-up (measured)": t_plain / t_spec,
        "same tokens": token_agreement(reference, out)[0],
    })
    add_to_summary("A4", f"speculative, from scratch, {prompt_name} prompt",
                   f"seconds for {n_new} tokens", t_plain, t_spec,
                   higher_is_better=False)
display(pd.DataFrame(spec_rows).set_index("prompt").round(2))

The same in `generate()`: `assistant_model=draft` (transformers picks and adapts the
number of draft tokens per round itself) and `prompt_lookup_num_tokens` (no draft model),
on both prompts. "same tokens" compares with the target alone. transformers may print
deprecation warnings from its own assisted-generation code; they do not affect the
results.

In [ ]:
SPEC_OPTIONS = {
    "target alone": {},
    "draft model": {"assistant_model": draft},
    "prompt lookup": {"prompt_lookup_num_tokens": cfg.prompt_lookup_tokens},
}

for prompt_name, ids in PROMPTS.items():
    # One run per option to check the output; it also serves as the warm-up
    outputs = {
        name: generate(target, ids, **opts) for name, opts in SPEC_OPTIONS.items()
    }
    a4_runs = {
        name: measure_generate(target, ids, n_warmup=0, **opts)
        for name, opts in SPEC_OPTIONS.items()
    }
    table = speed_table(a4_runs)
    table["same tokens"] = [
        token_agreement(outputs["target alone"], out)[0] for out in outputs.values()
    ]
    log.info(f"{prompt_name} prompt:")
    display(table)
    add_runs_to_summary("A4", a4_runs, f", {prompt_name} prompt")

In [ ]:
del draft, SPEC_OPTIONS  # SPEC_OPTIONS also holds the draft model
free_memory()

# Summary

Every comparison above, against the baseline of its own section. "x better": 2 means
twice as fast or half the memory; below 1 means worse. The confidence intervals are in
each section's table.

In [ ]:
summary = pd.DataFrame(SUMMARY)
display(summary.set_index(["section", "technique", "metric"]).round(3))

rows = summary.iloc[::-1]  # first comparison at the top of the chart
y = np.arange(len(rows))
x_better = rows["x better"].to_numpy()
fig, ax = plt.subplots(figsize=(10, 0.35 * len(rows) + 1.5), layout="constrained")
# Bars run from 1 (same as the baseline) to the measured factor. Log scale: 2x better
# and 2x worse get the same length
ax.barh(y, x_better - 1, left=1, height=0.6,
        color=np.where(x_better >= 1, BETTER, WORSE))
for yi, xb in zip(y, x_better):
    ax.annotate(f"{xb:.2f}x", (xb, yi), xytext=(4 if xb >= 1 else -4, 0),
                textcoords="offset points", ha="left" if xb >= 1 else "right",
                va="center", fontsize=8)
labels = rows["section"] + "  " + rows["technique"] + ": " + rows["metric"]
ax.set_yticks(y, labels, fontsize=8)
ax.set_xscale("log")
ax.axvline(1, color="#c3c2b7", linewidth=1)
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
ax.set(xlabel="x better than the section's baseline (log scale; below 1 = worse)",
       title="Every comparison against its own baseline")
plt.show()

What to look for. This is the pattern expected from the literature, not a result; your
numbers are in the table above.
- **A1**: the largest gain, and it grows with the output length (O(n^2) against O(n)).
- **A2**: compilation removes launch overhead, so it helps most where overhead is a big
  part of a step: small models and fast GPUs. The one-off compile time only pays off for
  fixed shapes and many tokens.
- **A3**: large memory and speed gains for long prompts, where eager may run out of
  memory. Close to 1x for short-prompt generation.
- **A4**: larger gains on the grounded prompt. Prompt lookup close to 1x on the open
  prompt. The output stays the target's own.

Pick by where the time goes: per-step overhead (A2), long prompts (A3), or one read of
the weights per token (A4). A1 is always on in practice.

In [ ]:
free_memory()
log.info("Done. Colab: Runtime > Disconnect and delete runtime. "
         "VM: stop it in the console.")